In [1]:
import json
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
from plot_progress import gather_metrics
from parse_levels import find_levels_in_configs, find_levels_in_configs_glob
import pandas as pd
import matplotlib.ticker as ticker
import re
from parse_levels import compute_gap_in_percentage, compute_gap_in_percentage_list, process_metrics, convert_to_dict
from parse_levels import human_train_time_dict
workspace_base_path = lambda item: os.path.join('/checkpoint/maui/zhaobc/scientist/workspace', item)

In [2]:
base_file_path = '../workspace_templates/nanogpt_speedrun/record_{i}/train_gpt2.py'
base_metrics = '../workspace_templates/nanogpt_speedrun/record_{i}/results.json'
l1_knowledge_base_path = '../data/nanogpt_speedrun_knowledge_in_levels/record_{i}/level_1_pseudo.txt'
l2_knowledge_base_path = '../data/nanogpt_speedrun_knowledge_in_levels/record_{i}/level_2_description.txt'
l5_knowledge_base_path = '../data/nanogpt_speedrun_knowledge_in_levels/record_{i}/level_5_paper.txt'

In [3]:
# final json structure
# {
#     "record_id": {
#         "human": {
#             "code": "...",
#             "metrics": {
#                 "val_loss": ...,
#                 "train_time": ...,
#             }
#         },
#         "next_human": {
#             "code": "...",
#             "metrics": {
#                 "val_loss": ...,
#                 "train_time": ...,
#             }
#         },
#         "ai_repro": {
#             "code": "...",
#             "metrics": {
#                 "val_loss": ...,
#                 "train_time": ...,
#             }
#         },
#         "ai_negative": {
#             "code": "...",
#             "metrics": {
#                 "val_loss": ...,
#                 "train_time": ...,
#             }
#         },
#     }
# }

In [4]:
final_json = {}
for record_id in range(1, 19):
    human_code = open(base_file_path.format(i=record_id), 'r').read()
    next_human_code = open(base_file_path.format(i=record_id+1), 'r').read()
    metrics = json.load(open(base_metrics.format(i=record_id), 'r'))
    next_metrics = json.load(open(base_metrics.format(i=record_id+1), 'r'))
    final_json[record_id] = {
        "human": {
            "code": human_code,
            "metrics": {
                "val_loss": metrics['metrics']['val_loss'],
                "train_time": metrics['metrics']['train_time'],
            }
        },
        "next_human": {
            "code": next_human_code,
            "metrics": {
                "val_loss": next_metrics['metrics']['val_loss'],
                "train_time": next_metrics['metrics']['train_time'],
            }
        }
    }

In [14]:
ori_results = find_levels_in_configs_glob(
    [
        '/checkpoint/maui/zhaobc/scientist/workspace/record_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/nanogpt_speedrun/record_*',
        # '/checkpoint/maui/zhaobc/scientist/old_workspace/nanogpt_speedrun/record_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250422_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250405_*'
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_1[2-8]_20250412_*',
        #  '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250408_*',
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250405_*'
        # '/checkpoint/maui/zhaobc/scientist/workspace/record_1[2-8]_20250409_*',
    ]
)

Found 894 directories


  0%|          | 0/894 [00:00<?, ?it/s]

100%|██████████| 894/894 [04:26<00:00,  3.36it/s]


In [15]:
with open('results.cache', 'w') as f:
    json.dump(ori_results, f, indent=4)

In [5]:
with open('results.cache', 'r') as f:
    ori_results = json.load(f)

In [6]:
from parse_levels import filter_folder_info

In [7]:

folder_info = ori_results

In [8]:
for level in [1, 12, 125]:
   for model in ['o3-mini', 'deepseek-r1']:
      tree_search = filter_folder_info(folder_info, [
         ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
         ('runner', 'bon'),
         ('model', model),
         ('n_initial_hypotheses', 1),
         ('n_hypotheses', 3),
      ])
      forest_search = filter_folder_info(folder_info, [
         ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
         ('runner', 'bon'),
         ('model', model),
         ('n_initial_hypotheses', 3),
         ('n_hypotheses', 3),
      ])

      ori_aide = filter_folder_info(folder_info, [
         ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
         ('runner', 'aide'),
         ('model', model),
         ('n_initial_hypotheses', 3),
         ('n_hypotheses', 1),
      ])

      multi_aide = filter_folder_info(folder_info, [
         ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
         ('runner', 'aide'),
         ('model', model),
         ('n_initial_hypotheses', 3),
         ('n_hypotheses', 3),
      ])

      flat = filter_folder_info(folder_info, [
         ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
         ('runner', 'bon'),
         ('model', model),
         ('n_initial_hypotheses', 50),
         # ('n_hypotheses', 1),
      ])

      print(f"for level {level}, model {model}")
      print(f"tree search: {len(tree_search)}")
      print(f"forest search: {len(forest_search)}")
      print(f"ori aide: {len(ori_aide)}")
      print(f"multi aide: {len(multi_aide)}")
      print(f"flat: {len(flat)}")
      print()
      

for level 1, model o3-mini
tree search: 18
forest search: 18
ori aide: 18
multi aide: 18
flat: 18

for level 1, model deepseek-r1
tree search: 18
forest search: 18
ori aide: 40
multi aide: 18
flat: 18

for level 12, model o3-mini
tree search: 18
forest search: 18
ori aide: 36
multi aide: 18
flat: 18

for level 12, model deepseek-r1
tree search: 18
forest search: 18
ori aide: 46
multi aide: 18
flat: 18

for level 125, model o3-mini
tree search: 18
forest search: 36
ori aide: 46
multi aide: 26
flat: 18

for level 125, model deepseek-r1
tree search: 18
forest search: 36
ori aide: 46
multi aide: 34
flat: 18



In [9]:
record_dict_w_level = {
    'tree': {},
    'forest': {},
    'ori_aide': {},
    'multi_aide': {},
    'flat': {},
}
for level in [12, 125]:
#    for model in ['o3-mini', 'deepseek-r1']:
    model = 'o3-mini'
    tree_search = filter_folder_info(folder_info, [
        ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
        ('runner', 'bon'),
        ('model', model),
        ('n_initial_hypotheses', 1),
        ('n_hypotheses', 3),
    ])
    forest_search = filter_folder_info(folder_info, [
        ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
        ('runner', 'bon'),
        ('model', model),
        ('n_initial_hypotheses', 3),
        ('n_hypotheses', 3),
    ])

    ori_aide = filter_folder_info(folder_info, [
        ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
        ('runner', 'aide'),
        ('model', model),
        ('n_initial_hypotheses', 3),
        ('n_hypotheses', 1),
    ])

    multi_aide = filter_folder_info(folder_info, [
        ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
        ('runner', 'aide'),
        ('model', model),
        ('n_initial_hypotheses', 3),
        ('n_hypotheses', 3),
    ])

    flat = filter_folder_info(folder_info, [
        ('levels', level),  # level 125 stands for using pseudo-code (1), text (2), and paper (5)
        ('runner', 'bon'),
        ('model', model),
        ('n_initial_hypotheses', 50),
        # ('n_hypotheses', 1),
    ])

    record_dict_w_level['tree'][level] = tree_search
    record_dict_w_level['forest'][level] = forest_search
    record_dict_w_level['ori_aide'][level] = ori_aide
    record_dict_w_level['multi_aide'][level] = multi_aide
    record_dict_w_level['flat'][level] = flat


In [10]:
record_dict_w_level['tree']

{12: {'record_12_20250429_084908_2142693-2142657-1': {'record': 12,
   'levels': [12],
   'ideator': 'dummy',
   'knowledge_coder': False,
   'runner': 'bon',
   'model': 'o3-mini',
   'n_initial_hypotheses': 1,
   'n_hypotheses': 3,
   'debug_prob': None},
  'record_1_20250429_084906_2142683-2142655-0': {'record': 1,
   'levels': [12],
   'ideator': 'dummy',
   'knowledge_coder': False,
   'runner': 'bon',
   'model': 'o3-mini',
   'n_initial_hypotheses': 1,
   'n_hypotheses': 3,
   'debug_prob': None},
  'record_6_20250429_084906_2142688-2142655-5': {'record': 6,
   'levels': [12],
   'ideator': 'dummy',
   'knowledge_coder': False,
   'runner': 'bon',
   'model': 'o3-mini',
   'n_initial_hypotheses': 1,
   'n_hypotheses': 3,
   'debug_prob': None},
  'record_17_20250429_084908_2142698-2142657-6': {'record': 17,
   'levels': [12],
   'ideator': 'dummy',
   'knowledge_coder': False,
   'runner': 'bon',
   'model': 'o3-mini',
   'n_initial_hypotheses': 1,
   'n_hypotheses': 3,
   'debu

In [11]:
# o3_bon = process_metrics(o3_bon)
# for search in ['tree', 'forest', 'ori_aide', 'multi_aide', 'flat']:
#     for level in [12, 125]:
#         record_dict_w_level[search][level] = process_metrics(record_dict_w_level[search][level])

In [12]:
for search in ['tree', 'forest', 'ori_aide', 'multi_aide', 'flat']:
    for level in [12, 125]:
        for record_folder, record_info in record_dict_w_level[search][level].items():
            # print(record_folder, record_info['record'])
            l1_knowledge_base_path = l1_knowledge_base_path.format(i=record_info['record'])
            l2_knowledge_base_path = l2_knowledge_base_path.format(i=record_info['record'])
            if level == 125:
                l5_knowledge_base_path = l5_knowledge_base_path.format(i=record_info['record'])
                knowledges = [l1_knowledge_base_path, l2_knowledge_base_path, l5_knowledge_base_path]
            elif level == 12:
                knowledges = [l1_knowledge_base_path, l2_knowledge_base_path]
            else:
                knowledges = [l1_knowledge_base_path]

            full_knowledge = ''
            for knowledge_path in knowledges:
                if os.path.exists(knowledge_path):
                    with open(knowledge_path, 'r') as f:
                        knowledge = f.read()
                        full_knowledge += '<li>' + knowledge + '</li>\n'

            record_dict_w_level[search][level][record_folder]['knowledge'] = full_knowledge

            # get human codes
            human_code = open(base_file_path.format(i=record_info['record']), 'r').read()
            next_human_code = open(base_file_path.format(i=record_info['record']+1), 'r').read()
            metrics = json.load(open(base_metrics.format(i=record_info['record']), 'r'))
            next_metrics = json.load(open(base_metrics.format(i=record_info['record']+1), 'r'))
            record_dict_w_level[search][level][record_folder]['human_code'] = human_code
            record_dict_w_level[search][level][record_folder]['next_human_code'] = next_human_code
            record_dict_w_level[search][level][record_folder]['metrics'] = metrics
            record_dict_w_level[search][level][record_folder]['next_metrics'] = next_metrics

            folder_path = workspace_base_path(record_folder)
            # Get all version folders
            version_folders = [d for d in os.listdir(folder_path) if d.startswith('v_')]

            # Get code and metrics for each version
            for v_folder in version_folders:
                version_num = int(v_folder.split('_')[1])
                try:
                    code_path = os.path.join(folder_path, v_folder, 'train_gpt2.py')
                    metrics_path = os.path.join(folder_path, v_folder, 'results.json')
                    
                    if os.path.exists(code_path) and os.path.exists(metrics_path):
                        with open(code_path, 'r') as f:
                            code = f.read()
                        with open(metrics_path, 'r') as f:
                            metrics = json.load(f)
                        
                        record_dict_w_level[search][level][record_folder][f'v_{version_num}'] = {
                            'code': code,
                            'metrics': {
                                'val_loss': metrics['metrics']['val_loss'],
                                'train_time': metrics['metrics']['train_time']
                            }
                        }
                except Exception as e:
                    print(f"Error processing record {record_id} version {version_num}: {str(e)}")
                    continue



In [13]:
with open('/checkpoint/maui/zhaobc/scientist/code_analysis_with_all_versions_knowledge.json', 'w') as f:
    json.dump(record_dict_w_level, f, indent=4)

In [14]:
with open('/checkpoint/maui/zhaobc/scientist/code_analysis_with_all_versions_knowledge.json', 'r') as f:
    record_dict_w_level = json.load(f)


In [15]:
record_dict_w_level['tree'] # different search methods
record_dict_w_level['tree']['12'] # different levels, 12 -> pseudo-code+text, 125 -> pseudo-code+text+paper
record_dict_w_level['tree']['12']['record_12_20250429_084908_2142693-2142657-1'] # different versions

{'record': 12,
 'levels': [12],
 'ideator': 'dummy',
 'knowledge_coder': False,
 'runner': 'bon',
 'model': 'o3-mini',
 'n_initial_hypotheses': 1,
 'n_hypotheses': 3,
 'debug_prob': None,
 'knowledge': '<li>\n# Pseudo Code Changes\n\n// --- Attention Mechanism Improvements ---\n// Dynamic attention window scaling replaces fixed 1024 token window\nFUNCTION document_causal_mask(blocksize):\n    RETURN mask WHERE:\n        (query_position >= key_position) AND                // Standard causal masking\n        (same_document) AND                                 // Document boundary constraints\n        (query_position - key_position < dynamic_blocksize) // Increasing context window\n\nDURING TRAINING:\n    // Linearly scale attention block size from 64 to 1792 tokens over training\n    current_step ← training_progress (0..1)\n    attn_blocksize ← 64 + (1792 - 64) * current_step\n    attn_blocksize ← ROUND_DOWN_TO_NEAREST_64(attn_blocksize)\n\n// --- Optimizer Configuration Updates ---\nADJ

In [28]:
with open('/checkpoint/maui/zhaobc/scientist/code_analysis_with_all_versions.json', 'w') as f:
    json.dump(final_json, f, indent=4)
